# 每日新闻摘要器

这是一个简单的每日新闻摘要工具，面向印度报纸《The Hindu》。它使用简易网页抓取，并用 llama3.2 模型生成摘要，作为第 2 天作业的一部分。



In [1]:
# 导入依赖
import requests
from openai import OpenAI
from IPython.display import HTML



### 抓取代码



In [9]:
from bs4 import BeautifulSoup

# 浏览器 User-Agent，降低被网站拒绝的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"}

def fetch_website_info(url):
    # 抓取网页正文，截断到 2000 字符
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content,"html.parser")
    title = soup.title.string if soup.title else "No title found"

    if soup.body:
        # 去掉脚本、样式与表单等无关节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]



In [10]:
# 检查本地 Ollama 是否在运行
requests.get("http://localhost:11434").content



b'Ollama is running'

In [11]:
# 通过 OpenAI 兼容接口连接本地 Ollama
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")



In [12]:
# 系统提示：要求以 HTML 输出多条新闻标题与简述
system_prompt = """You are an assistant journalist. You are tasked with summarizing any important news headline in the current day's newspaper. Respond in html.  Summarize the info in separate headings and paragraphs. Heading would be the headline and a small description of it as the paragraph. Skip any advertisements or description about the newpaper or anything that might not be news"""

# 用户提示：至少包含 8 条标题
user_prompt = """Here is the website information for the newspaper, the hindu. Include atleast 8 headlines"""




In [13]:
def headline_summarizer(url):
    # 抓取网页 → 拼消息 → 调用模型 → 以 HTML 展示
    summary = fetch_website_info(url)
    messages = [{"role":"system", "content":system_prompt},
            {"role":"user", "content":user_prompt + summary}]
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return HTML(response.choices[0].message.content)



In [ ]:
# 对 The Hindu 首页做摘要
url = "https://www.thehindu.com/"
headline_summarizer(url)

